In [ ]:
import jaxmapse
import classy
import numpy as np
import matplotlib.pyplot as plt
# Enable full LaTeX power
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
})

In [ ]:
import jax.numpy as jnp

In [ ]:
emu = jaxmapse.trained_emulators["mnuw0wacdm_class"]

In [ ]:
k = emu.linear_pmm.k_grid

In [ ]:
params = np.array([0.29888 ,  3.044, 0.9665, 67.1734 , 0.022773 , 0.15  ,   0.06 , -1.5    , 0.42523])

cosmo = jaxmapse.w0waCDMCosmology(
        ln10As=params[1], 
        ns=params[2], 
        h=params[3] / 100.0,
        omega_b=params[4], 
        omega_c=params[5], 
        m_nu=params[6],
        w0=params[7], 
        wa=params[8], 
    )
D = cosmo.D_z(params[0])

In [ ]:
emu.linear_pkcb.out_minmax.shape

In [ ]:
# Linear spectra only. Do not call emu.get_Pk(...) here: that evaluates the Boost component,
# whose k-grid does not match the 300-point PCA linear spectra in this artifact.
Pcb = emu.get_linear_pkcb(params[1:], params[0], D)
Pmm = emu.get_linear_pmm(params[1:], params[0], D)


In [ ]:
plt.loglog(k, Pmm)
plt.loglog(k, Pcb)

In [ ]:
z = params[0]
h = params[3] / 100.0

cosmo_params = {
    "output": "mPk",
    "non_linear": "halofit",
    "P_k_max_h/Mpc": 50.0 / h,
    "z_pk": "0.0,5.",
    "h": h,
    "omega_b": params[4],
    "omega_cdm": params[5],
    "ln10^{10}A_s": params[1],
    "n_s": params[2],
    "tau_reio": 0.0568,
    "N_ur": 2.033,
    "N_ncdm": 1,
    "m_ncdm": params[6],
    "use_ppf": "yes",
    "w0_fld": params[7],
    "wa_fld": params[8],
    "fluid_equation_of_state": "CLP",
    "cs2_fld": 1.0,
    "Omega_Lambda": 0.0,
    "Omega_scf": 0.0,
    "halofit_min_k_max": 1000.0,
}

# =============================================================================
# OPTIMIZED k-GRID
# =============================================================================
K_MIN = 1e-5
K_BAO_START = 1e-2
K_BAO_END = 0.5
K_MAX = 20.0

# 1. Sparse low-k tail (smooth, linear in log-log)
_k_low  = np.geomspace(K_MIN, K_BAO_START, 15)

# 2. Dense BAO region (highly oscillatory)
_k_mid  = np.geomspace(K_BAO_START, K_BAO_END, 50)

# 3. Sparse high-k tail (smooth damping)
_k_high = np.geomspace(K_BAO_END, K_MAX, 15)

# Concatenate, dropping the overlapping boundary points [:-1]
k_grid = np.concatenate((_k_low[:-1], _k_mid[:-1], _k_high))

cosmo = classy.Class()
cosmo.set(cosmo_params)
cosmo.compute()

# P(k) in Mpc^3 units (k in Mpc^-1)
pk_lin_mm = np.array([cosmo.pk_lin(ki, z) for ki in k])
pk_lin_cb = np.array([cosmo.pk_cb_lin(ki, z) for ki in k])
pk_nl_mm = np.array([cosmo.pk(ki, z) for ki in k])

cosmo.struct_cleanup()
cosmo.empty()

In [ ]:
plt.loglog(k, pk_lin_cb)
plt.loglog(k, pk_lin_mm)
plt.loglog(k, pk_nl_mm)

In [ ]:
plt.loglog(k, pk_lin_cb)
plt.loglog(k, Pcb)
plt.xlabel(r'$k \ [1/\mathrm{Mpc}]$', fontsize=12)

In [ ]:
plt.plot(k, 100 * (1 - pk_lin_cb / Pcb))

# Axis Labels using LaTeX
plt.xlabel(r'$k \ [1/\mathrm{Mpc}]$', fontsize=12)
plt.ylabel(r'$100 \times (1 - \mathrm{emu}/\mathrm{class})$', fontsize=12)

# Scales and Reference Lines
plt.xscale('log')
plt.axhline(0, color='black', linestyle='-', linewidth=0.8) # Baseline
plt.axhline(0.5, color='gray', linestyle='--')             # +0.5% threshold
plt.axhline(-0.5, color='gray', linestyle='--')            # -0.5% threshold

plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

In [ ]:
plt.loglog(k, pk_lin_mm)
plt.loglog(k, Pmm)
plt.xlabel(r'$k \ [1/\mathrm{Mpc}]$', fontsize=12)
plt.ylabel(r'$100 \times (1 - \mathrm{emu}/\mathrm{class})$', fontsize=12)

In [ ]:
plt.plot(k, 100 * (1 - pk_lin_mm / Pmm))

# Axis Labels using LaTeX
plt.xlabel(r'$k \ [1/\mathrm{Mpc}]$', fontsize=12)
plt.ylabel(r'$100 \times (1 - \mathrm{emu}/\mathrm{class})$', fontsize=12)

# Scales and Reference Lines
plt.xscale('log')
plt.axhline(0, color='black', linestyle='-', linewidth=0.8) # Baseline
plt.axhline(0.5, color='gray', linestyle='--')             # +0.5% threshold
plt.axhline(-0.5, color='gray', linestyle='--')            # -0.5% threshold

plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

## Native HMCode2020 comparison: CAMB, CLASS, and full jaxmapse

This section compares three nonlinear HMCode predictions on the same output grid:

1. CAMB native HMCode2020 feedback.
2. CLASS native HMCode2020 feedback.
3. Full jaxmapse: emulated linear `Pmm`/`Pcb` passed to the JAX-native HMCode2020 kernel.

Important unit convention: the HMCode-trained jaxmapse artifact stores linear spectra in CLASS physical units,

\[
    k_{\rm artifact}\,[1/{\rm Mpc}], \qquad P_{\rm artifact}\,[{\rm Mpc}^3].
\]

The CAMB/CLASS HMCode reference below is evaluated in h-units, so before feeding the emulated spectra to HMCode we convert

\[
    k_h = k_{\rm artifact}/h, \qquad P_h = P_{\rm artifact}\,h^3.
\]


In [ ]:
import os
import contextlib
import time

import camb
from camb import model
import jax
from jaxmapse import HMCodeCosmology, hmcode_pmm_jax

jax.config.update("jax_enable_x64", True)


@contextlib.contextmanager
def silence_fds():
    """Silence verbose CLASS debug output in notebooks."""
    devnull = os.open(os.devnull, os.O_WRONLY)
    old_stdout = os.dup(1)
    old_stderr = os.dup(2)
    try:
        os.dup2(devnull, 1)
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(old_stdout, 1)
        os.dup2(old_stderr, 2)
        os.close(old_stdout)
        os.close(old_stderr)
        os.close(devnull)


In [ ]:
# Use the HMCode-trained artifact because it has both linear Pmm/Pcb and nonlinear HMCode metadata.
hmcode_emu = jaxmapse.trained_emulators["trained_mapse_class_hmcode_mnuw0waOkcdm"]

# Parameter order for this artifact:
# [ln10As, ns, H0, ombh2, omch2, Mnu, w0, wa, Omega_k, log10T_AGN]
p_hmcode = jnp.array([3.044, 0.9649, 67.36, 0.02237, 0.12, 0.06, -1.0, 0.0, 0.0, 7.8])
ln10As, ns_hm, H0, ombh2, omch2, Mnu, w0_hm, wa_hm, Omega_k, log10T_AGN = map(float, p_hmcode)

h_hm = H0 / 100.0
As = np.exp(ln10As) * 1.0e-10
Omega_b = ombh2 / h_hm**2
Omega_nu = Mnu / (93.14 * h_hm**2)
Omega_m = Omega_b + omch2 / h_hm**2 + Omega_nu
Omega_fld = 1.0 - Omega_m

hm_cosmo = HMCodeCosmology(
    Omega_m=Omega_m,
    Omega_b=Omega_b,
    h=h_hm,
    n_s=ns_hm,
    sigma_8=0.8108,  # HMCode container compatibility; sigma8 is inferred from P(k) internally.
    w0=w0_hm,
    wa=wa_hm,
    Omega_nu=Omega_nu,
    Omega_k=Omega_k,
)

# Practical grid used in the benchmark discussion.
z_hm = np.linspace(0.0, 3.5, 30)
k_out_h = np.geomspace(1.0e-4, 10.0, 128)  # h/Mpc

# The artifact linear grid is in 1/Mpc. Convert to h/Mpc for HMCode.
k_artifact = np.asarray(hmcode_emu.linear_pmm.k_grid)
k_support_h_full = k_artifact / h_hm
support_mask = (k_support_h_full >= 1.0e-4) & (k_support_h_full <= k_support_h_full[-1])
k_support_h = k_support_h_full[support_mask]

print(f"support grid: {len(k_support_h)} points, {k_support_h[0]:.3e} .. {k_support_h[-1]:.3e} h/Mpc")
print(f"output grid:  {len(k_out_h)} points, {k_out_h[0]:.3e} .. {k_out_h[-1]:.3e} h/Mpc")
print(f"redshifts:    {len(z_hm)} points, {z_hm[0]:.2f} .. {z_hm[-1]:.2f}")


In [ ]:
def relative_stats(x, y, k, band):
    mask = (k >= band[0]) & (k <= band[1])
    err = np.abs(x[:, mask] - y[:, mask]) / np.abs(y[:, mask])
    return np.max(err), np.mean(err), np.quantile(err, 0.95)


def print_relative_stats(label, x, y, k):
    print(label)
    for band in [(0.1, 1.0), (1.0e-3, 10.0)]:
        mx, mn, p95 = relative_stats(x, y, k, band)
        print(f"  {band}: max={mx:.3e}, mean={mn:.3e}, p95={p95:.3e}")


def camb_reference_grids(z, k_support_h, k_out_h):
    pars = camb.CAMBparams()
    pars.set_cosmology(
        H0=H0,
        ombh2=ombh2,
        omch2=omch2,
        mnu=Mnu,
        num_massive_neutrinos=1,
    )
    pars.InitPower.set_params(As=As, ns=ns_hm)
    pars.set_dark_energy(w=w0_hm, wa=wa_hm)
    pars.set_matter_power(redshifts=list(z), kmax=1000.0)
    pars.NonLinear = model.NonLinear_both
    pars.NonLinearModel.set_params(
        halofit_version="mead2020_feedback",
        HMCode_logT_AGN=log10T_AGN,
    )

    lin_mm = camb.get_matter_power_interpolator(
        pars, nonlinear=False, hubble_units=True, k_hunit=True,
        kmax=1000.0, zmax=float(np.max(z)), var1="delta_tot", var2="delta_tot",
    )
    lin_cb = camb.get_matter_power_interpolator(
        pars, nonlinear=False, hubble_units=True, k_hunit=True,
        kmax=1000.0, zmax=float(np.max(z)), var1="delta_nonu", var2="delta_nonu",
    )
    nl_mm = camb.get_matter_power_interpolator(
        pars, nonlinear=True, hubble_units=True, k_hunit=True,
        kmax=1000.0, zmax=float(np.max(z)), var1="delta_tot", var2="delta_tot",
    )

    pmm = np.vstack([lin_mm.P(float(zi), k_support_h) for zi in z])
    pcb = np.vstack([lin_cb.P(float(zi), k_support_h) for zi in z])
    pnl = np.vstack([nl_mm.P(float(zi), k_out_h) for zi in z])
    return pmm, pcb, pnl


def class_reference_grids(z, k_support_h, k_out_h):
    class_params = {
        "output": "mPk",
        "non linear": "hmcode",
        "hmcode_version": "2020_baryonic_feedback",
        "log10T_heat_hmcode": log10T_AGN,
        "P_k_max_1/Mpc": 1000.0,
        "z_max_pk": 5.0,
        "A_s": As,
        "n_s": ns_hm,
        "H0": H0,
        "omega_b": ombh2,
        "omega_cdm": omch2,
        "N_ur": 2.0328,
        "N_ncdm": 1,
        "m_ncdm": Mnu,
        "Omega_fld": Omega_fld,
        "w0_fld": w0_hm,
        "wa_fld": wa_hm,
    }
    with silence_fds():
        cc = classy.Class()
        cc.set(class_params)
        cc.compute()

    def grid(k_h, nonlinear=False, cb=False):
        out = np.empty((len(z), len(k_h)))
        with silence_fds():
            for iz, zi in enumerate(z):
                for ik, kh in enumerate(k_h):
                    k_mpc = kh * h_hm
                    if cb:
                        val = cc.pk_cb_lin(float(k_mpc), float(zi))
                    elif nonlinear:
                        val = cc.pk(float(k_mpc), float(zi))
                    else:
                        val = cc.pk_lin(float(k_mpc), float(zi))
                    out[iz, ik] = val * h_hm**3  # Mpc^3 -> (Mpc/h)^3
        return out

    pmm = grid(k_support_h, nonlinear=False, cb=False)
    pcb = grid(k_support_h, nonlinear=False, cb=True)
    pnl = grid(k_out_h, nonlinear=True, cb=False)
    sigma8, sigma8_cb = cc.sigma8(), cc.sigma8_cb()
    with silence_fds():
        cc.struct_cleanup()
        cc.empty()
    print(f"CLASS sigma8={sigma8:.6f}, sigma8_cb={sigma8_cb:.6f}")
    return pmm, pcb, pnl


def jax_hmcode_from_linear(pmm_support_h, pcb_support_h):
    return np.asarray(
        jax.block_until_ready(
            hmcode_pmm_jax(
                hm_cosmo,
                jnp.asarray(z_hm),
                jnp.asarray(k_out_h),
                jnp.asarray(k_support_h),
                jnp.asarray(pmm_support_h),
                jnp.asarray(pcb_support_h),
                nM=96,
                include_feedback=True,
            )
        )
    )


In [ ]:
# Native CAMB and CLASS references.
pmm_camb, pcb_camb, pnl_camb_native = camb_reference_grids(z_hm, k_support_h, k_out_h)
pmm_class, pcb_class, pnl_class_native = class_reference_grids(z_hm, k_support_h, k_out_h)

# Full jaxmapse linear predictions, converted from artifact physical units to h-units.
growth_cosmo = jaxmapse.w0waCDMCosmology(
    ln10As=ln10As,
    ns=ns_hm,
    h=h_hm,
    omega_b=ombh2,
    omega_c=omch2,
    m_nu=Mnu,
    w0=w0_hm,
    wa=wa_hm,
    omega_k=Omega_k * h_hm**2,
)
D_hm = growth_cosmo.D_z(jnp.asarray(z_hm))

pmm_emu_physical = np.asarray(jax.block_until_ready(hmcode_emu.get_linear_pmm(p_hmcode, jnp.asarray(z_hm), D_hm)))
pcb_emu_physical = np.asarray(jax.block_until_ready(hmcode_emu.get_linear_pkcb(p_hmcode, jnp.asarray(z_hm), D_hm)))

# Artifact convention: k in 1/Mpc, P in Mpc^3. HMCode convention here: k in h/Mpc, P in (Mpc/h)^3.
pmm_emu_h = pmm_emu_physical[:, support_mask] * h_hm**3
pcb_emu_h = pcb_emu_physical[:, support_mask] * h_hm**3

# JAX HMCode with exact CAMB/CLASS linear inputs and with emulated linear inputs.
pnl_jax_from_camb = jax_hmcode_from_linear(pmm_camb, pcb_camb)
pnl_jax_from_class = jax_hmcode_from_linear(pmm_class, pcb_class)
pnl_full_jaxmapse = jax_hmcode_from_linear(pmm_emu_h, pcb_emu_h)


In [ ]:
print("Linear spectra on the common support grid")
print_relative_stats("emulated Pmm vs CAMB Pmm", pmm_emu_h, pmm_camb, k_support_h)
print_relative_stats("emulated Pmm vs CLASS Pmm", pmm_emu_h, pmm_class, k_support_h)
print_relative_stats("CLASS Pmm vs CAMB Pmm", pmm_class, pmm_camb, k_support_h)
print_relative_stats("emulated Pcb vs CAMB Pcb", pcb_emu_h, pcb_camb, k_support_h)
print_relative_stats("emulated Pcb vs CLASS Pcb", pcb_emu_h, pcb_class, k_support_h)
print_relative_stats("CLASS Pcb vs CAMB Pcb", pcb_class, pcb_camb, k_support_h)

print("\nNonlinear HMCode spectra on k_out")
print_relative_stats("CLASS native HMCode vs CAMB native HMCode", pnl_class_native, pnl_camb_native, k_out_h)
print_relative_stats("JAX HMCode(CAMB linear) vs CAMB native HMCode", pnl_jax_from_camb, pnl_camb_native, k_out_h)
print_relative_stats("JAX HMCode(CLASS linear) vs CLASS native HMCode", pnl_jax_from_class, pnl_class_native, k_out_h)
print_relative_stats("full jaxmapse HMCode vs CAMB native HMCode", pnl_full_jaxmapse, pnl_camb_native, k_out_h)
print_relative_stats("full jaxmapse HMCode vs CLASS native HMCode", pnl_full_jaxmapse, pnl_class_native, k_out_h)
print_relative_stats("full jaxmapse HMCode vs JAX HMCode(CLASS linear)", pnl_full_jaxmapse, pnl_jax_from_class, k_out_h)


In [ ]:
# Representative nonlinear ratios at a few redshifts.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, z_plot in zip(axes, [0.0, 1.0, 3.5]):
    iz = int(np.argmin(np.abs(z_hm - z_plot)))
    ax.axhline(0.0, color="black", lw=0.8)
    ax.plot(k_out_h, 100.0 * (pnl_class_native[iz] / pnl_camb_native[iz] - 1.0), label="CLASS native / CAMB native")
    ax.plot(k_out_h, 100.0 * (pnl_jax_from_class[iz] / pnl_class_native[iz] - 1.0), label="JAX HMCode(CLASS lin) / CLASS native")
    ax.plot(k_out_h, 100.0 * (pnl_full_jaxmapse[iz] / pnl_class_native[iz] - 1.0), label="full jaxmapse / CLASS native")
    ax.set_xscale("log")
    ax.set_xlabel(r"$k\ [h/\mathrm{Mpc}]$")
    ax.set_title(fr"$z={z_hm[iz]:.2f}$")
    ax.grid(True, which="both", alpha=0.2)
axes[0].set_ylabel(r"$100\times(P/P_{\rm ref}-1)$")
axes[-1].legend(fontsize=8)
plt.tight_layout()
plt.show()
